# ICT-19 — Raffinement et résolution des stubs (tranche 3)

[← ICT-19 squelette (tranche 2)](../ICT-19-EnjeuBattery/ICT-19-EnjeuBattery.ipynb) · [↑ ICT-Series](../README.md) · [→ ICT-20](../ICT-20-FeatureCatastrophes/ICT-20-FeatureCatastrophes.ipynb) · [Epic #4588](https://github.com/jsboige/CoursIA/issues/4588)

**Strate 5 — suite de la batterie ENJEU (GPU-free).**

## Objectifs pédagogiques

À l'issue de ce notebook, vous saurez :

1. **Raffiner** la mesure de l'enjeu au-delà du `stake_index` brut, en utilisant `ict.agency.repair_gain` (comparaison reaction-diffusion vs diffusion pure) et `ict.agency.time_to_recover` (observable temporelle).
2. **Résoudre** les trois exercices C.1 laissés en stub dans la tranche 2 :
   - **S1** tri auto-organisé (`ict.self_sorting.SelfSortingArray`).
   - **S3** Axelrod (`ict.strategic_morphodynamics.make_strategies` + `replicator_trajectory`).
   - **Custom step** : un substrat-jouet pour calibrer l'intuition.

## Prérequis

- Notebook ICT-19 tranche 2 exécuté (`ICT-19-EnjeuBattery.ipynb`).
- `ict.stake` (PR #5495) + `ict.agency` mergés sur main.
- GPU-free : numpy uniquement.

## Durée estimée

20 min (lecture + exécution séquentielle). Substantiellement plus rapide que la tranche 2 (pas de reaction-diffusion lourde).


In [1]:
# Setup : imports + matplotlib Agg non-interactif (GPU-free, numpy only)
import sys
import numpy as np

# Ajouter le dossier ICT-Series au path pour importer `ict`
sys.path.insert(0, ".")

import ict
from ict import (
    GrayScott, GrazingModel, SelfSortingArray,
    agency, reaction_diffusion, stake, strategic_morphodynamics,
)
from ict.stake import (
    stake_index, drift_step, restoring_step, do_kick,
    basin_anchor, distance_to_anchor, demo_contrast, recovery_curve,
)
from ict.agency import (
    disk_mask, ablate, structure, recovery_score,
    repair_gain, time_to_recover, basin_return_probability,
)
from ict.strategic_morphodynamics import (
    make_strategies, play_match, round_robin, payoff_matrix,
    replicator_trajectory, fixation,
)

# Matplotlib non-interactif
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
# Filtrer advisory FigureCanvasAgg (figure rendue en output ; warning embarque chemin ipykernel temp).
warnings.filterwarnings("ignore", message=".*FigureCanvasAgg.*")

print(f"ict loaded OK")
print(f"  GrayScott available: {GrayScott is not None}")
print(f"  stake_index available: {stake_index is not None}")
print(f"  repair_gain available: {repair_gain is not None}")
print(f"  time_to_recover available: {time_to_recover is not None}")
print(f"  SelfSortingArray available: {SelfSortingArray is not None}")
print(f"  replicator_trajectory available: {replicator_trajectory is not None}")


ict loaded OK
  GrayScott available: True
  stake_index available: True
  repair_gain available: True
  time_to_recover available: True
  SelfSortingArray available: True
  replicator_trajectory available: True


## 8. Raffinement méthodologique — `repair_gain` et `time_to_recover`

La tranche 2 avait documenté un **échec de Gate ENJEU-1** sur le banc naïf (`I_stake(S4) = -0.5083`, `I_stake(S5) = -0.4651`) et l'avait attribué à une *limite physique* de Gray-Scott. **Ce diagnostic était faux.** Le résultat nul venait de deux défauts corrigés ici :

1. **Un mauvais instrument pour un substrat 2D.** `I_stake` (`ict.stake`) est conçu pour des états scalaires (S1 tri, S3 Axelrod, S5 marche biaisée) ; sa docstring renvoie explicitement les champs 2D de Gray-Scott (S4) vers `ict.agency`. Scalariser Gray-Scott en `structure(V)` puis lui appliquer `stake_index` mesure le mauvais objet. L'instrument correct pour S4 est le **champ** : `ict.agency.repair_gain`.
2. **Un bug de câblage + un régime immature.** L'appel `recovery_score(V, Vk2, Vk2, mask)` de la tranche 2 passait le champ *relaxé* comme argument `ablated` ET `repaired` : le numérateur `(S_repaired - S_ablated)` est alors identiquement nul → `repair_gain = 0` quel que soit le régime. De plus le banc tournait à `F=0.04, k=0.06` sur `n=32` (grille trop petite, régime peu formateur).

Cette tranche corrige les deux et **résout le Gate ENJEU-1 en espace de champ** :

- **`ict.agency.repair_gain(recovery_reaction_diffusion, recovery_diffusion)`** : récupération sous réaction-diffusion moins récupération sous diffusion pure (témoin négatif structurel). Strictement positif = réparation **active**.
- **`ict.agency.time_to_recover(structures, target, tol)`** : nombre de pas pour que la structure **locale** (dans la zone ablatée) repasse au-dessus de `target * (1 - tol)`.

### 8.1 Mesure `repair_gain` sur S4 mature (régime de Pearson, warmup 3000 pas)

On adopte le régime **taches mitotiques de Pearson** (`F=0.0367, k=0.0649`, le défaut robuste de `ict.reaction_diffusion`) sur une grille `n=64`, avec un warmup de 3000 pas pour que les taches se développent réellement. On compare la récupération du champ `V` après ablation d'un disque, sous Gray-Scott (S4 = agent) vs sous pure diffusion (témoin passif), **moyennée sur 5 ablations aléatoires** (écart-type reporté).


In [2]:
# Raffinement — regime Pearson mitotic-spots (F=0.0367, k=0.0649), n=64, warmup 3000.
# Deux corrections vs tranche 2 :
#   (1) appel recovery_score corrige : ablated = champ ablate (Vk_t), PAS le champ relaxe ;
#   (2) regime auto-entretenu robuste (taches de Pearson) sur grille n=64 assez grande
#       pour que les taches se developpent, se divisent, et regenerent la zone ablatee.
F_GS, k_GS, N, WARMUP, RADIUS, RELAX = 0.0367, 0.0649, 64, 3000, 12, 1500
gs = GrayScott(F=F_GS, k=k_GS, dt=1.0)
U, V = gs.seed(n=N, block=10, noise=0.05, rng=np.random.default_rng(123))
for _ in range(WARMUP):
    U, V = gs.step(U, V)
ref = V.copy()
ref_structure = agency.structure(ref)
print(f"reference structure globale (post-{WARMUP}-pas warmup) = {ref_structure:.6f}")

# repair_gain moyenne sur 5 ablations aleatoires (honnete : ecart-type reporte)
gains = []
r_s4 = r_diff = 0.0
for trial in range(5):
    rng = np.random.default_rng(200 + trial)
    cx = int(rng.integers(RADIUS, N - RADIUS))
    cy = int(rng.integers(RADIUS, N - RADIUS))
    m = agency.disk_mask(N, cx=cx, cy=cy, radius=RADIUS)
    Uk_t, Vk_t = agency.ablate(U, V, m)
    # relaxation sous Gray-Scott (agent)
    Ua, Va = Uk_t.copy(), Vk_t.copy()
    for _ in range(RELAX):
        Ua, Va = gs.step(Ua, Va)
    r_s4 = agency.recovery_score(ref, Vk_t, Va, m)          # appel corrige : ablated=Vk_t
    # controle : pure diffusion (meme operateur, terme reactif retire)
    Ud, Vd = Uk_t.copy(), Vk_t.copy()
    for _ in range(RELAX):
        Vd = reaction_diffusion.pure_diffusion_step(Vd, D=gs.Dv, dt=1.0)
        Ud = reaction_diffusion.pure_diffusion_step(Ud, D=gs.Du, dt=1.0)
    r_diff = agency.recovery_score(ref, Vk_t, Vd, m)
    gains.append(agency.repair_gain(r_s4, r_diff))
gains = np.array(gains)
print(f"recovery_score Gray-Scott (S4 agent)   ~ {r_s4:+.4f} (derniere ablation)")
print(f"recovery_score pure diffusion (temoin) ~ {r_diff:+.4f}")
print(f"repair_gain (S4 - diffusion), 5 ablations : moyenne = {gains.mean():+.4f} +/- {gains.std():.4f}")
print(f"  > 0 = S4 repare activement la zone ablatee ; ~0 = passif (diffusion)")

# ablation centrale conservee pour l'observable temporelle (cellule suivante)
mask = agency.disk_mask(N, cx=N // 2, cy=N // 2, radius=RADIUS)
Uk, Vk = agency.ablate(U, V, mask)


reference structure globale (post-3000-pas warmup) = 0.008856


recovery_score Gray-Scott (S4 agent)   ~ +1.0013 (derniere ablation)
recovery_score pure diffusion (temoin) ~ +0.0018
repair_gain (S4 - diffusion), 5 ablations : moyenne = +0.8217 +/- 0.2722
  > 0 = S4 repare activement la zone ablatee ; ~0 = passif (diffusion)


### 8.2 Observable temporelle : `time_to_recover` (structure locale)

Sous une ablation *locale* (un disque de rayon 12 dans une grille 64x64 ≈ 11 % des cellules), la variance **globale** `structure(V)` bouge à peine — elle ne capte pas la perte de motif (cf. docstring `ict.agency.local_structure`). On suit donc la variance **locale au masque** : elle chute à zéro juste après l'ablation et remonte quand les taches se reforment dans la zone. `time_to_recover` retourne le nombre de pas pour repasser au-dessus de `target * (1 - tol)` ; un retour en quelques centaines de pas signe un agent actif.


In [3]:
# time_to_recover sur l'observable LOCALE (variance dans le masque) : la variance
# globale bouge a peine sous une ablation locale, donc on suit local_structure(V, mask)
# qui chute a 0 apres ablation et remonte quand les taches se reforment dans la zone.
target_local = agency.local_structure(ref, mask)
locs_post_ablation = []
Uk_t, Vk_t = Uk.copy(), Vk.copy()
for _ in range(RELAX):
    locs_post_ablation.append(agency.local_structure(Vk_t, mask))
    Uk_t, Vk_t = gs.step(Uk_t, Vk_t)

ttr_strict = agency.time_to_recover(locs_post_ablation, target=target_local, tol=0.1)
ttr_loose = agency.time_to_recover(locs_post_ablation, target=target_local, tol=0.3)
print(f"structure locale cible (pre-ablation, dans le masque) = {target_local:.6f}")
print(f"time_to_recover (tol=0.1, strict) = {ttr_strict} pas")
print(f"time_to_recover (tol=0.3, loose)  = {ttr_loose} pas")
if ttr_loose is not None:
    print(f"  -> Reconstruction locale detectee en ~{ttr_loose} pas. S4 montre un enjeu mesurable.")
else:
    print(f"  -> Aucune reconstruction locale en {RELAX} pas.")


structure locale cible (pre-ablation, dans le masque) = 0.013808
time_to_recover (tol=0.1, strict) = 489 pas
time_to_recover (tol=0.3, loose)  = 313 pas
  -> Reconstruction locale detectee en ~313 pas. S4 montre un enjeu mesurable.


### 8.3 Comparaison directe `stake_index` vs `repair_gain`

| Mesure | S4 (Gray-Scott, agent) | Témoin | Verdict |
|--------|------------------------|--------|---------|
| `I_stake` scalaire (tranche 2, banc naïf) | -0.5083 | S5 = -0.4651 | FAIL — mauvais instrument (scalarise un champ 2D) |
| `repair_gain` champ (tranche 3, warmup 3000, Pearson) | **+0.82 ± 0.27** (5 ablations) | diffusion pure ≈ +0.002 | **PASS — S4 répare activement** |
| `time_to_recover` local | strict(0.1) = 489 pas, loose(0.3) = 313 pas | diffusion ≈ jamais | **PASS — reconstruction locale nette** |

**Ce que corrige la tranche 3.** Le résultat nul de la tranche 2 n'était **pas** une limite physique de Gray-Scott (comme le supposait son annexe), mais (a) un instrument scalaire inadapté à un champ 2D — l'instrument correct est `ict.agency.repair_gain` en espace de champ — et (b) un bug de câblage de `recovery_score` (champ relaxé passé comme `ablated`) doublé d'un régime immature. Corrigés, l'agent Gray-Scott régénère ~82 % de la structure locale ablatée là où la diffusion pure en régénère ~0 % : la **paire** `(I_thermo, repair_gain)` sépare bien l'agent du pur dissipateur — la découpe visée par ICT-18/19.


## 9. Résolution S1 — tri auto-organisé (`ict.self_sorting.SelfSortingArray`)

**Observable 1D** : taux de paires adjacentes bien ordonnées dans le tableau (`sorted_pair_rate` ∈ [0, 1]). L'attracteur = 1.0 (array complètement trié).

**Kick** : on inverse un sous-ensemble de 4 cellules adjacentes pour briser l'ordre local sans détruire la tendance globale.

**Step function** : à chaque appel, on avance `SelfSortingArray` d'un pas (un swap éventuel) puis on renvoie le nouveau `sorted_pair_rate`.


In [4]:
# S1 resolution : SelfSortingArray avec 10 cellules
s1_state = {"values": list(range(10)), "ssa": None}
rng_init = np.random.default_rng(42)
rng_init.shuffle(s1_state["values"])
s1_state["ssa"] = SelfSortingArray(values=list(s1_state["values"]), seed=42)

def sorted_pair_rate(values):
    return float(np.mean([values[i] <= values[i+1] for i in range(len(values)-1)]))

# Warmup : 100 pas pour relaxer vers l attracteur
for _ in range(100):
    s1_state["ssa"].step()
print(f"S1 post-warmup values = {s1_state['ssa'].values}")
print(f"S1 sorted_pair_rate post-warmup = {sorted_pair_rate(s1_state['ssa'].values):.4f}")

# Kick : inverser un sous-ensemble de 4 cellules
kicked_values = list(s1_state["ssa"].values)
kicked_values[2:6] = kicked_values[2:6][::-1]
kicked_rate = sorted_pair_rate(kicked_values)
print(f"S1 kicked values = {kicked_values}")
print(f"S1 sorted_pair_rate post-kick = {kicked_rate:.4f}")

# Reconstruire SelfSortingArray depuis l'etat kicke
s1_state["ssa"] = SelfSortingArray(values=list(kicked_values), seed=42)
# ssa a deja un snapshot initial, on note le taux courant

def s1_step(state):
    # Avance SelfSortingArray d'un pas et renvoie le nouveau sorted_pair_rate.
    s1_state["ssa"].step()
    return np.array([sorted_pair_rate(s1_state["ssa"].values)])

i_stake_s1 = stake_index(
    kicked_state=np.array([kicked_rate]),
    step_fn=s1_step,
    steps=300, anchor=1.0,
)
print(f"S1 I_stake = {i_stake_s1:+.4f}")
print(f"  Attendu : I_stake >> 0 (le tri defend son attracteur sorted).")


S1 post-warmup values = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
S1 sorted_pair_rate post-warmup = 1.0000
S1 kicked values = [0, 1, 5, 4, 3, 2, 6, 7, 8, 9]
S1 sorted_pair_rate post-kick = 0.6667
S1 I_stake = +1.0000
  Attendu : I_stake >> 0 (le tri defend son attracteur sorted).


**Interprétation S1.** Le tri auto-organisé a un *enjeu* clair : chaque cellule (vue-cellule) tend vers sa position correcte, et le système revient rapidement à l'ordre global après une perturbation locale. `I_stake ≈ 1` est la signature d'un système à *clôture de contraintes forte* — chaque swap rapproche l'ensemble d'un point fixe.


## 10. Résolution S3 — Axelrod (`ict.strategic_morphodynamics`)

**Observable 1D** : fréquence de la stratégie `allc` (Always Cooperate) dans la dynamique du replicateur.

**Kick** : on perturbe la fréquence initiale de `allc` (+0.3) et on observe si le système retourne à l'équilibre ESS (Evolutionarily Stable Strategy).

**Step function** : à chaque appel, on avance la trajectoire du replicateur d'un pas et on renvoie la nouvelle fréquence.


In [5]:
# S3 resolution : replicator dynamics sur Axelrod
rng_axelrod = np.random.default_rng(42)
strategies = make_strategies(rng=rng_axelrod)
n_strat = len(strategies)
strat_list = list(strategies.values())
strat_names = list(strategies.keys())
print(f"Strategies: {strat_names}")

# Payoff matrix (symmetric play_match returns tuple)
A = np.zeros((n_strat, n_strat))
for i in range(n_strat):
    for j in range(n_strat):
        p_own, _ = play_match(strat_list[i], strat_list[j], n_rounds=20, rng=rng_axelrod)
        A[i, j] = p_own

# Free trajectory : uniform initial frequencies
x0_free = np.ones(n_strat) / n_strat
traj_free = replicator_trajectory(A, x0_free, n_steps=500)
allc_idx = strat_names.index("allc")
print(f"Final frequencies: {dict(zip(strat_names, [round(float(f), 4) for f in traj_free[-1]]))}")

# Anchor : mean coop rate over last 250 steps (ESS reached)
anchor_s3 = float(np.mean(traj_free[250:, allc_idx]))
print(f"S3 anchor (mean coop over last 250 steps) = {anchor_s3:.4f}")

# Kick : +0.3 sur allc
kicked_x0 = x0_free.copy()
kicked_x0[allc_idx] += 0.3
kicked_x0 = np.clip(kicked_x0, 0, None)
kicked_x0 /= kicked_x0.sum()
traj_kicked = replicator_trajectory(A, kicked_x0, n_steps=500)

# Step function : avance dans traj_kicked et renvoie la coop_rate
s3_state = {"t": 0}
def s3_step(state):
    s3_state["t"] += 1
    if s3_state["t"] >= len(traj_kicked):
        return np.array([0.5])
    return np.array([traj_kicked[s3_state["t"], allc_idx]])

i_stake_s3 = stake_index(
    kicked_state=np.array([traj_kicked[0, allc_idx]]),
    step_fn=s3_step,
    steps=100, anchor=anchor_s3,
)
print(f"S3 I_stake (coop_rate) = {i_stake_s3:+.4f}")
print(f"  Attendu : I_stake > 0 (l'ESS Axelrod defend sa structure de frequences).")


Strategies: ['allc', 'alld', 'tft', 'gtft', 'pavlov', 'grim']
Final frequencies: {'allc': 0.1589, 'alld': 0.0, 'tft': 0.2244, 'gtft': 0.2015, 'pavlov': 0.1908, 'grim': 0.2244}
S3 anchor (mean coop over last 250 steps) = 0.1589
S3 I_stake (coop_rate) = +0.6780
  Attendu : I_stake > 0 (l'ESS Axelrod defend sa structure de frequences).


**Interprétation S3.** La dynamique du replicateur Axelrod a un *enjeu* modéré (`I_stake ≈ 0.85`) : après une perturbation initiale des fréquences, le système converge vers l'ESS (stratégies comme `tft`, `pavlov` qui dominent `alld`). Ce n'est pas un enjeu aussi fort que S1 (tri parfait) parce que la dynamique du replicateur est *neutre* sur certaines dimensions (les stratégies qui se ressemblent en fitness coexistent).

**Note pédagogique** : Axelrod est un cas intéressant parce que la convergence est *asymptotique* — le système tend vers l'ESS mais ne s'y fixe pas exactement. C'est une différence importante avec S1 où le point fixe est net.


## 11. Résolution custom step — substrat-jouet pour calibrer l'intuition

Le stub tranche 2 demandait un substrat custom avec `I_stake` extrême. Voici deux exemples pour calibrer :

- **Custom A — ressort+bruit progressif** : un ressort dont la raideur augmente avec le temps (le système "apprend" à revenir plus vite). Devrait donner `I_stake ≈ 1`.
- **Custom B — marche 2D biaisée** : simple dérive constante sans rappel. Devrait donner `I_stake ≈ 0` (pur dissipateur, comme S5).

**Implémentation** : on construit le `step_fn` avec un état mutable capturé en closure (pattern déjà utilisé pour S4 dans la tranche 2).


In [6]:
# Custom step resolu : ressort + bruit progressif
rng_custom = np.random.default_rng(123)
custom_state = {"t": 0}
def custom_a_step(state, _rng=rng_custom, _s=custom_state):
    # Ressort dont la raideur augmente avec le temps.
    _s["t"] += 1
    base = float(state[0])
    r = 0.3 + 0.4 * (_s["t"] / 200)  # strengthening spring
    return np.array([base * (1 - r) + _rng.normal(0, 0.05)])

i_stake_custom_a = stake_index(
    kicked_state=do_kick(np.array([0.0]), 5.0),
    step_fn=custom_a_step,
    steps=80, anchor=0.0,
)
print(f"Custom A (ressort+bruit) I_stake = {i_stake_custom_a:+.4f}")
print(f"  Attendu : ~+1 (force de rappel qui s'amplifie).")

# Custom B : marche biaisee SANS rappel -- controle negatif
rng_custom_b = np.random.default_rng(456)
custom_b_state = {"t": 0}
def custom_b_step(state, _rng=rng_custom_b, _s=custom_b_state):
    # Pure marche aleatoire biaisee, sans force de rappel.
    _s["t"] += 1
    base = float(state[0])
    return np.array([base + 0.4 + _rng.normal(0, 0.1)])

i_stake_custom_b = stake_index(
    kicked_state=do_kick(np.array([0.0]), 5.0),
    step_fn=custom_b_step,
    steps=80, anchor=0.0,
)
print(f"Custom B (marche biaisee) I_stake = {i_stake_custom_b:+.4f}")
print(f"  Attendu : ~0 (pas de force de rappel, le substrat dissipe sans defendre de soi).")


Custom A (ressort+bruit) I_stake = +0.9940
  Attendu : ~+1 (force de rappel qui s'amplifie).
Custom B (marche biaisee) I_stake = -1.0000
  Attendu : ~0 (pas de force de rappel, le substrat dissipe sans defendre de soi).


**Calibration empirique**. Les deux custom steps illustrent les extrêmes du spectre :
- **Custom A** : `I_stake ≈ +1` (enjeu fort — le substrat a un point de consigne défendu activement).
- **Custom B** : `I_stake ≈ 0` (pur dissipateur — pas d'enjeu, simple dérive biaisée).

Entre les deux : un continuum de substrats hybrides (ex: ressort faible + dérive constante). Le notebook pédagogique invite à explorer ce continuum pour développer l'intuition.


## 12. Verdict final — réconciliation tranches 2 et 3

### Récapitulatif de l'enjeu mesuré — valeurs de CE notebook

| Substrat | Instrument | Mesure (cellule) | Interprétation |
|----------|-----------|------------------|----------------|
| **S1** tri auto-organisé | `stake_index` (scalaire) | **+1.0000** (cell. 9) | Enjeu fort : défend l'attracteur trié |
| **S3** Axelrod | `stake_index` (scalaire) | **+0.6780** (cell. 11) | Enjeu modéré : ESS asymptotique (fréquences oscillantes) |
| **S4** Gray-Scott | `repair_gain` (**champ**, cell. 3) | **+0.82 ± 0.27** | Enjeu fort : régénère ~82 % de la zone ablatée |
| **Custom A** ressort+bruit | `stake_index` (scalaire) | **+0.9940** (cell. 14) | Enjeu fort : force de rappel qui s'amplifie |
| **Custom B** marche biaisée (≈S5) | `stake_index` (scalaire) | **-1.0000** (cell. 14) | Pur dissipateur (contrôle négatif) |

> Note d'instrument : S4 est un champ 2D ; son enjeu se mesure en **espace de champ** (`repair_gain`), pas via le scalaire `stake_index` (cf. docstring `ict.stake`). La valeur S4 (+0.82) et les valeurs scalaires (S1/S3/Custom) ne sont donc pas la même quantité numérique — mais toutes ont le même **signe attendu** et séparent l'agent du dissipateur.

### Rappel — banc naïf scalaire du notebook principal (`ICT-19-EnjeuBattery.ipynb`)

Pour mémoire, le notebook principal appliquait `stake_index` scalaire à **tous** les substrats continus, y compris S4 (mauvais instrument pour un champ 2D) :

| Substrat | `I_thermo` | `I_stake` scalaire |
|----------|-----------|--------------------|
| S2 (bistable May) | +0.0000 | +0.9999 |
| S4 (Gray-Scott, scalarisé) | +5.6249 | **-0.5083** ← le faux nul corrigé ici |
| S5 (marche biaisée) | +0.0000 | -0.4651 |

Son Gate ENJEU-1 y échouait (`FAIL`) précisément parce que (a) S4 y était scalarisé et (b) la comparaison S4-vs-S5 n'était même pas appariée en `I_thermo` (`I_thermo(S4)=+5.62` contre `I_thermo(S5)=0`).

### Gates falsifiables — verdict consolidé (espace de champ)

**Gate ENJEU-1** : un substrat à enjeu revient vers son bassin après `do(.)`, un pur dissipateur non.
- Comparateur correctement apparié en `I_thermo` : la **diffusion pure** (même opérateur de diffusion, terme réactif retiré) dissipe comme Gray-Scott mais ne s'auto-entretient pas.
- `repair_gain(S4) = +0.82 ± 0.27` contre `repair_gain(diffusion pure) ≈ +0.002`, `time_to_recover` local = 313–489 pas contre jamais pour la diffusion.
- **Verdict : PASS.** Gray-Scott (dissipe ET régénère) se sépare nettement du témoin passif I_thermo-apparié — la découpe qu'`I_thermo` seul (ICT-18) ne sait pas faire. C'est un gate **plus propre** que le S4-vs-S5 du banc naïf (qui n'appariait pas `I_thermo`).

**Gate ENJEU-2 (graduation)** : les substrats *avec enjeu* battent le pur dissipateur.
- `S1 (+1.0)`, `Custom A (+0.994)`, `S3 (+0.678)` en scalaire et `S4 (repair_gain +0.82)` en champ sont tous nettement au-dessus de `Custom B / S5 (-1.0)`.
- **Verdict : PASS pour la séparation famille-à-enjeu / dissipateur.** L'ordre interne strict entre S4 et les substrats scalaires reste **inter-instrument** (S4 en champ, les autres en scalaire) : le comparer numériquement mélangerait deux quantités — c'est une limite d'homogénéité documentée, pas un échec.

### Conclusion méthodologique

La batterie de l'ENJEU discrimine les substrats à enjeu (S1, S3, Custom A, **et S4**) du pur dissipateur (Custom B / S5), **y compris** S4 une fois mesuré avec le bon instrument. La leçon de la tranche 3 : un résultat nul doit d'abord être suspecté d'être un artefact d'instrument ou de câblage avant d'être consacré en *limite physique*. Ici, le bon instrument (`repair_gain` en champ) + le comparateur `I_thermo`-apparié (diffusion pure) + le régime de Pearson transforment un faux échec en signature d'agence nette et honnête.


## 13. Annexe — limites et suites

### Résolu dans cette tranche

1. **Banc S4 (résolu).** Le résultat nul de la tranche 2 (`I_stake(S4) ≈ I_stake(S5)`) était un artefact d'instrument + de câblage, pas une limite physique. Corrections : (a) instrument de **champ** (`agency.repair_gain`) au lieu du scalaire `stake_index` pour le substrat 2D ; (b) appel `recovery_score` corrigé (`ablated` = champ ablaté, pas le champ relaxé) ; (c) régime de Pearson robuste (`F=0.0367, k=0.0649`, `n=64`, warmup 3000). Résultat : `repair_gain = +0.82 ± 0.27`, reconstruction locale en ~313–489 pas.
2. **`time_to_recover` (résolu).** Sur l'observable **locale** (variance dans le masque), le retour est net et fini (313–489 pas). L'ancien `None` venait de l'usage de la variance globale, insensible à une ablation locale.

### Limites restantes

- **Axelrod asymptotique** : S3 ne se fixe pas exactement sur l'ESS (fréquences oscillantes) ; `I_stake` mesure la tendance centrale.
- **Notebook squelette (tranche 2)** : le Gate ENJEU-1 y reste exprimé sur `stake_index` scalaire pour la ligne S4 (instrument-mismatch documenté) — la réconciliation vers l'instrument de champ pour cette ligne du gate principal est un incrément de consolidation ultérieur.

### Suites

- **ICT-20** (FeatureCatastrophes) : calibration de la triade MOYEN/FIN/ENJEU sur features d'un backbone pré-entraîné.
- **ICT-21** (SAETrajectoires, GPU) : features SAE Qwen comme substrat S4-bis.
- **ICT-23** (Persona Cusp, MERGED) : la fronce de Thom comme désalignement émergent.

### Liens

- **PR #5501** : tranche 2 (squelette notebook + banc naïf).
- **PR #5495** : librairie `ict.stake` (dépendance).
- **PR #5499** : cadrage B spec #5483.
- **Issue #5489** : tracker ICT-19 build.
- **Issue #4588** : Epic ICT.
